# STaR SFT — Qwen3.8-27B

Self-Taught Reasoner fine-tuning: extract ALL analysis turns from winning games (≥1 level completed)
across multiple Qwen3.8 public run artifacts, then QLoRA-SFT the model on those winning trajectories.

**Run order:** Cell 1 (mine) → Cell 2 (inspect) → Cell 3 (download model if needed) → Cell 4 (load) → Cell 5 (train) → Cell 6 (save)

In [1]:
# ── Cell 1: Mine STaR dataset ─────────────────────────────────────────────────
import json, re, glob
from pathlib import Path
from datetime import date

REPO = Path('..').resolve()

# Sources: foysalemonshanto (Qwen3.8), wuliao0 (Qwen3.8 Anim), keithtyser (Flash Next)
SOURCES = [
    (str(REPO / 'artifacts'), 'foysalemonshanto'),
    ('/tmp/qwen38-wuliao/artifacts', 'wuliao0'),
    ('/tmp/qwen38-keith/artifacts', 'keithtyser'),
]

TODAY = date.today().isoformat().replace('-', '')
OUT = REPO / f'results/star_sft_qwen38_{TODAY}.jsonl'
OUT.parent.mkdir(parents=True, exist_ok=True)

SECTION_RE = re.compile(
    r'\n(\[(?:SYSTEM PROMPT|THINKING|ANALYZER STATUS)\])\n'
)

def parse_transcript(tr: str, event: dict):
    """Return SFT record {system, user, assistant} or None if parse fails."""
    parts = SECTION_RE.split(tr)
    system_prompt = thinking = ''
    for i, p in enumerate(parts):
        if p == '[SYSTEM PROMPT]' and i + 1 < len(parts):
            system_prompt = parts[i + 1].strip()
        elif p == '[THINKING]' and i + 1 < len(parts):
            thinking = parts[i + 1]
            # Strip trailing [ANALYZER STATUS] block
            thinking = re.sub(r'\n?\[ANALYZER STATUS\].*$', '', thinking,
                              flags=re.DOTALL).strip()
    if not system_prompt or not thinking:
        return None

    level      = event.get('level', '?')
    action_num = event.get('action_num', '?')
    step       = event.get('analysis_step', '?')
    board_ascii = event.get('board_ascii', '')

    user_msg = (
        f"Level {level} | analysis step {step} | actions taken: {action_num}\n\n"
        f"Current board:\n{board_ascii}\n\n"
        "Analyse the game state and take the best next action(s)."
    )

    return {
        'system':        system_prompt,
        'user':          user_msg,
        'assistant':     thinking,
        'level':         level,
        'action_num':    action_num,
        'analysis_step': step,
    }


records = []
skipped = 0

for source_dir, source_name in SOURCES:
    files = sorted(glob.glob(f'{source_dir}/*events.jsonl'))
    src_wins = 0
    for f in files:
        try:
            events = [json.loads(l) for l in open(f, errors='replace') if l.strip()]
        except Exception as e:
            print(f'  skip {f}: {e}')
            continue
        game_id = Path(f).name.split('-')[0]
        has_win = any(e.get('level_completed') for e in events)
        if not has_win:
            continue

        # STaR: all analysis turns in games that eventually won a level
        for e in events:
            if e.get('type') != 'analysis' or not e.get('transcript'):
                continue
            rec = parse_transcript(e['transcript'], e)
            if rec is None:
                skipped += 1
                continue
            rec['game_id'] = game_id
            rec['source']  = source_name
            records.append(rec)
            src_wins += 1

    print(f'  {source_name}: {src_wins} turns mined')

print(f'\nTotal: {len(records)} STaR turns ({skipped} skipped).')

with open(OUT, 'w') as f:
    for r in records:
        f.write(json.dumps(r) + '\n')
print(f'Saved → {OUT}')

  foysalemonshanto: 690 turns mined
  wuliao0: 1074 turns mined
  keithtyser: 989 turns mined

Total: 2753 STaR turns (59 skipped).
Saved → /home/lavolpe/Bureau/Kaggle/ARC-AGI-3/results/star_sft_qwen38_20260923.jsonl


In [2]:
# ── Cell 2: Inspect dataset ───────────────────────────────────────────────────
from collections import Counter

records = [json.loads(l) for l in open(OUT) if l.strip()]

by_source = Counter(r['source'] for r in records)
by_game   = Counter(r['game_id'] for r in records)

asst_lens = [len(r['assistant'].split()) for r in records]
user_lens = [len(r['user'].split()) for r in records]

print(f'Total records   : {len(records)}')
print(f'By source       : {dict(by_source)}')
print(f'Games covered   : {len(by_game)} → {dict(sorted(by_game.items()))}')
print(f'Assistant words : min={min(asst_lens)} median={sorted(asst_lens)[len(asst_lens)//2]} max={max(asst_lens)}')
print(f'User words      : min={min(user_lens)} median={sorted(user_lens)[len(user_lens)//2]} max={max(user_lens)}')

# Show one sample
r = records[0]
print(f'\n--- Sample (game={r["game_id"]} step={r["analysis_step"]}) ---')
print(f'System (first 200): {r["system"][:200]}')
print(f'User (first 200):   {r["user"][:200]}')
print(f'Assistant (first 400): {r["assistant"][:400]}')

Total records   : 2753
By source       : {'foysalemonshanto': 690, 'wuliao0': 1074, 'keithtyser': 989}
Games covered   : 20 → {'ar25': 142, 'bp35': 109, 'cd82': 164, 'cn04': 123, 'ft09': 140, 'ka59': 146, 'lf52': 162, 'lp85': 142, 'ls20': 147, 'm0r0': 100, 'r11l': 139, 're86': 163, 's5i5': 142, 'sb26': 157, 'sc25': 145, 'su15': 151, 'tr87': 104, 'tu93': 106, 'vc33': 132, 'wa30': 139}
Assistant words : min=43 median=679 max=6306
User words      : min=86 median=86 max=86

--- Sample (game=ar25 step=1) ---
System (first 200): You are a coding agent solving a grid-based puzzle game.

Game overview:
- You are solving a multi-level grid puzzle game. 
- You are called repeatedly over the course of a run. Treat each turn as one
User (first 200):   Level 1 | analysis step 1 | actions taken: 0

Current board:
bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbSSSbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbY
bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbSSSbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbY
bbbbbbbbb
Assistant (first 400): Now I can see the shapes c

In [ ]:
# ── Cell 3: Acquire Qwen3.8-27B, properly dequantized to BF16 ────────────────
# CRITICAL FIX (2026-09-24): the raw FP8 checkpoint (qwen38_27b_fp8) stores
# weights natively as float8_e4m3fn with a companion weight_scale_inv tensor
# per 128x128 block -- true_value = fp8_value * block_scale. Loading it by
# deleting quantization_config and casting straight to bf16 (the old approach)
# SILENTLY SKIPS this rescaling. Verified: the resulting model loads with no
# NaN but generates pure gibberish even on a trivial prompt ("Hello, how are
# you?" -> Chinese/garbage tokens). Every training run before this fix
# (local AND the ~23h Colab A100 run) fine-tuned a LoRA on top of these
# mis-scaled weights and must be discarded.
#
# Fix: notebooks/export_bf16_qwen38.py dequantizes properly (reused the
# existing dequantize_fp8 logic from export_bf16.py, written earlier for
# Ministral's identical FP8 block-quant format) and writes a real BF16
# checkpoint. Verified fix: same "Hello" prompt now generates a coherent
# response. Run that script once if qwen38_27b_bf16/ doesn't exist yet.
import os, subprocess
from pathlib import Path

MODEL_DIR = Path('/home/lavolpe/Bureau/Kaggle/ARC-AGI-3/checkpoints/qwen38_27b_bf16')
FP8_DIR   = Path('/home/lavolpe/Bureau/Kaggle/ARC-AGI-3/checkpoints/qwen38_27b_fp8')

if MODEL_DIR.exists() and any(MODEL_DIR.glob('*.safetensors')):
    shard_count = len(list(MODEL_DIR.glob('*.safetensors')))
    total_gb = sum(p.stat().st_size for p in MODEL_DIR.glob('*.safetensors')) / 1e9
    print(f'Dequantized BF16 model already present: {shard_count} shards, {total_gb:.1f} GB')
else:
    if not (FP8_DIR.exists() and any(FP8_DIR.glob('*.safetensors'))):
        print('Downloading Qwen3.8-27B FP8 (~30 GB)...')
        FP8_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ['kaggle', 'datasets', 'download',
             'driessmit1/qwen3-8-27b-fp8-hf-017b9c7a',
             '-p', str(FP8_DIR), '--unzip'],
            check=True
        )
        print('Download complete.')
    print('Dequantizing FP8 -> BF16 (needs ~54GB disk, a few minutes)...')
    subprocess.run(['python3', str(Path('..') / 'notebooks' / 'export_bf16_qwen38.py')], check=True)
    shard_count = len(list(MODEL_DIR.glob('**/*.safetensors')))
    print(f'Dequantization complete. Shards: {shard_count}')

In [4]:
# ── Cell 3b: Download (uncomment & run if Cell 3 says model not found) ────────
# MODEL_DIR.mkdir(parents=True, exist_ok=True)
# !kaggle datasets download driessmit1/qwen3-8-27b-fp8-hf-017b9c7a -p {MODEL_DIR} --unzip

In [ ]:
# ── Cell 4: Load Qwen3.8 (corrected BF16) with BnB NF4 ───────────────────────
# Requires `pip install flash-linear-attention` (fla) -- without it, this
# model's linear_attn layers fall back to a pure-PyTorch reference kernel that
# OOMs almost immediately (verified: crashed inside torch_chunk_gated_delta_rule
# on the very first layer at seq_len=6144-8192). With fla installed, the model
# auto-picks the fused kernel and gets through a full forward pass instead.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

def _find_model_root(base: Path) -> Path:
    if (base / 'config.json').exists():
        return base
    for p in sorted(base.rglob('config.json')):
        return p.parent
    raise FileNotFoundError(f'config.json not found under {base}')

MODEL_ROOT = _find_model_root(MODEL_DIR)
print(f'Loading from: {MODEL_ROOT}')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_ROOT))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# export_bf16_qwen38.py already removed quantization_config from this
# checkpoint's config.json (real dequantized bf16 weights, nothing to strip).
cfg = AutoConfig.from_pretrained(str(MODEL_ROOT), trust_remote_code=True)

print('Loading model (NF4)...')
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_ROOT),
    config=cfg,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    # Real prompts run 5000-6400 tokens (measured) -- eager attention
    # materializes a full LxL score matrix per head/layer, which would be
    # enormous at this length. SDPA uses a fused memory-efficient kernel.
    attn_implementation='sdpa',
)

free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f'Model loaded. VRAM free: {free_gb:.1f} GB')

# Sanity check -- NaN check alone is NOT enough (verified: mis-scaled FP8
# weights loaded without NaN but generated pure gibberish). Actually look at
# the generated text, not just isnan().
dummy = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': 'Hello, how are you? Answer in one short sentence.'}],
    tokenize=False, add_generation_prompt=True,
)
dummy_inputs = tokenizer(dummy, return_tensors='pt').to(model.device)
with torch.no_grad():
    dummy_out = model.generate(**dummy_inputs, max_new_tokens=30, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
sanity_text = tokenizer.decode(dummy_out[0][dummy_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'Sanity generation: {sanity_text!r}')
print('^ must be coherent English, not gibberish/foreign-script tokens -- if it looks broken, '
      'the checkpoint is mis-scaled again, do not proceed to training.')

In [6]:
# ── Cell 5: Prepare dataset for SFT ──────────────────────────────────────────
import random
from torch.utils.data import Dataset
from tqdm.auto import tqdm

# VERIFIED on this exact GPU (RTX 3090 24GB) with a standalone repro script:
# 8192 and 6144 both OOM during backward even with fla + logits_to_keep +
# attn-only LoRA (Cell 6) + empty_cache (Cell 7) -- the 27B NF4 model alone
# already uses ~17GB, leaving only ~5.5GB for activations/backward. 5632 was
# the largest length that got within ~130MB of fitting in repeated tests.
# Real prompts are 5059-6418 tokens (measured) -- there is NO length that
# guarantees 100% dataset coverage AND fits this card; this trades some
# coverage (samples with prompt > 5632 get skipped by the filter below) for
# the best shot at actually fitting. If it still OOMs, this needs a bigger GPU
# (e.g. train on Kaggle instead of locally), not further shrinking.
MAX_LENGTH = 5632
SEED = 42
random.seed(SEED)

records = [json.loads(l) for l in open(OUT) if l.strip()]
random.shuffle(records)

class StarSFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.samples = []
        skipped = 0
        for r in tqdm(records, desc='Tokenizing', unit='sample'):
            messages = [
                {'role': 'system',    'content': r['system']},
                {'role': 'user',      'content': r['user']},
                {'role': 'assistant', 'content': r['assistant']},
            ]
            try:
                full_text = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=False
                )
                prompt_msgs = messages[:-1]
                prompt_text = tokenizer.apply_chat_template(
                    prompt_msgs, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                skipped += 1
                continue

            enc = tokenizer(
                full_text,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            )
            if enc['input_ids'].shape[1] < 20:
                skipped += 1
                continue

            prompt_len = tokenizer(
                prompt_text,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            )['input_ids'].shape[1]

            ids    = enc['input_ids'][0]
            attn   = enc['attention_mask'][0]
            labels = ids.clone()
            labels[:prompt_len] = -100

            if (labels != -100).sum().item() == 0:
                # prompt alone (system+user) consumed the whole max_length
                # budget -- no assistant tokens survive truncation, nothing
                # for this sample to teach the model.
                skipped += 1
                continue

            self.samples.append({
                'input_ids':      ids,
                'attention_mask': attn,
                'labels':         labels,
            })

        print(f'Dataset: {len(self.samples)} samples ({skipped} skipped)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


dataset = StarSFTDataset(records, tokenizer, MAX_LENGTH)

lens = [s['input_ids'].shape[0] for s in dataset]
print(f'Token lengths: min={min(lens)} median={sorted(lens)[len(lens)//2]} max={max(lens)}')
truncated = sum(1 for l in lens if l == MAX_LENGTH)
print(f'Truncated at {MAX_LENGTH}: {truncated}/{len(lens)}')

Tokenizing:   0%|          | 0/2753 [00:00<?, ?sample/s]

Dataset: 1231 samples (1522 skipped)
Token lengths: min=5262 median=5632 max=5632
Truncated at 5632: 1189/1231


In [7]:
# ── Cell 6: Configure LoRA ────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, TaskType

# prepare_model_for_kbit_training casts all bf16 params (incl. FP8 scale tensors)
# to float32, which OOMs on a 24 GB card after loading 27B NF4.
# Manual equivalent: enable gradient checkpointing + input grads only.
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

# Attention-only (no mlp gate/up/down_proj): measured ~900MB-1.2GB lower peak
# forward memory than adapting all 7 projections, for a small quality tradeoff.
# Needed headroom given how tight this card is at the prompt lengths this
# dataset requires (see Cell 5).
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 10,485,760 || all params: 26,906,484,224 || trainable%: 0.0390


In [8]:
# ── Cell 7: Train ─────────────────────────────────────────────────────────────
import torch
import torch.nn.functional as F
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

ADAPTER_DIR = REPO / f'checkpoints/star_qwen38_{TODAY}'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

n_eval = max(1, len(dataset) // 10)
train_ds = torch.utils.data.Subset(dataset, range(len(dataset) - n_eval))
eval_ds  = torch.utils.data.Subset(dataset, range(len(dataset) - n_eval, len(dataset)))
print(f'Train: {len(train_ds)}  Eval: {len(eval_ds)}')

collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

# Chunking the fp32 upcast (previous version) didn't actually help: autograd
# retains every chunk's buffers for backward anyway, so total backward memory
# was the same as one big upcast -- explains why 8192->6144 barely moved the
# OOM. The real hog is projecting the 151936-vocab LM head over the WHOLE
# sequence, when ~80%+ of every sequence is the masked (-100) prompt that
# contributes nothing to the loss. Qwen3.5's forward supports `logits_to_keep`
# to only materialize logits for a suffix of positions -- use it to skip the
# prompt entirely (batch size is 1, so "the assistant span" is unambiguous).
class BF16LossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')  # [1, L]
        seq_len = labels.shape[1]
        valid_mask = labels[0] != -100
        nz = valid_mask.nonzero()

        if nz.numel() == 0:
            # Fully-masked sample (Cell 5 filters these, but guard anyway):
            # cheap forward just to keep the loss connected to the graph.
            outputs = model(**inputs, logits_to_keep=1)
            loss = outputs.logits.sum() * 0.0
            return (loss, outputs) if return_outputs else loss

        first_valid = int(nz[0].item())
        # Keep logits from one position BEFORE the first valid label onward --
        # that position's hidden state is what predicts the first assistant
        # token. Never materialize vocab-sized logits for the ~5000-6400 prompt
        # tokens before it.
        logits_to_keep = seq_len - first_valid + 1

        outputs = model(**inputs, logits_to_keep=logits_to_keep)
        logits = outputs.logits  # [1, logits_to_keep, V], bf16 -- small now

        shift_logits = logits[:, :-1, :].float()
        shift_labels = labels[:, first_valid:].to(shift_logits.device)

        loss = F.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=-100,
        )

        if return_outputs:
            # Eval path (prediction_step needs a real outputs object) --
            # skip the aggressive cache-clearing below, eval isn't as tight.
            return loss, outputs

        # Forward leaves very little headroom on this card at these sequence
        # lengths (verified: often <400MB free). Release cached-but-unused
        # allocator blocks before backward needs to grow further.
        del outputs, logits, shift_logits
        torch.cuda.empty_cache()
        return loss


EPOCHS   = 3
BATCH    = 1
GRAD_ACC = 8
WARMUP_STEPS = max(1, int(len(train_ds) / GRAD_ACC * EPOCHS * 0.05))

args = TrainingArguments(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP_STEPS,
    fp16=False,
    bf16=True,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='epoch',
    save_total_limit=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim='paged_adamw_8bit',
    report_to='none',
    seed=SEED,
    dataloader_pin_memory=False,
)

trainer = BF16LossTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
)

print(f'warmup_steps={WARMUP_STEPS}  Starting STaR SFT training...')
trainer.train()
print('Training complete.')


Train: 1108  Eval: 123
warmup_steps=20  Starting STaR SFT training...


/home/lavolpe/Bureau/Kaggle/ARC-AGI-3/.venv/lib/python3.12/site-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
[W923 11:16:57.973443327 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 135593984, total: 25288769536).
[W923 11:16:57.063937663 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (free: 135593984, total: 25288769536).
[W923 11:16:57.066216548 CUDACachingAllocator.cpp:528] expandable_segments: memory mapping failed with OOM on device 0 while trying to map 20971520 bytes (f

OutOfMemoryError: CUDA out of memory. Tried to allocate 170.00 MiB. GPU 0 has a total capacity of 23.55 GiB of which 169.31 MiB is free. Process 3037 has 262.00 MiB memory in use. Including non-PyTorch memory, this process has 22.31 GiB memory in use. Of the allocated memory 21.79 GiB is allocated by PyTorch, and 212.25 MiB is reserved by PyTorch but unallocated.

In [ ]:
# ── Cell 8: Save adapter + quick sanity check ─────────────────────────────────
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f'Adapter saved → {ADAPTER_DIR}')

# List what was saved
for p in sorted(ADAPTER_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size // 1024} KB)')

# Sanity: generate one response with the adapter
model.eval()
prompt = tokenizer.apply_chat_template(
    [{'role': 'system', 'content': 'You are a coding agent solving a grid-based puzzle game.'},
     {'role': 'user',   'content': 'Level 1, action 0. Board:\nBBBBBB\nB....B\nB....B\nBBBBBB\n\nAnalyse and act.'}],
    tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f'\nSample response (first 500):\n{resp[:500]}')

## Next steps after training

1. **Upload adapter** to Kaggle as a private dataset:
   ```bash
   kaggle datasets create -p checkpoints/star_qwen38_YYYYMMDD -t "STaR SFT Qwen3.8 adapter"
   ```

2. **Switch kernel model** — update `dataset_sources` in `build_tuning_variant.py`:
   - Replace `driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot` → `driessmit1/qwen3-8-27b-fp8-hf-017b9c7a`
   - Add the adapter dataset
   - Add a LEVER that loads the adapter at inference time (Cell 13 hook)

3. **If NaN in Cell 4** — run the FP8→BF16 export first (same approach as `notebooks/export_bf16.py`
   used for Ministral), then reload from the BF16 checkpoint.